# Streaming Pipeline — Apache Kafka + Spark Structured Streaming

Este notebook implementa la ingesta en tiempo real exigida por la guía del proyecto
(sección 3.1 del PDF oficial).

**Arquitectura**:

```
Productor sintético (Python)  →  Kafka Topic (Confluent Cloud)  →  Spark Structured Streaming  →  bronze.bronze_bookings_stream (Delta)
```

## Setup previo en Confluent Cloud (una sola vez)

1. Crear cuenta gratis en https://confluent.cloud (5 GB/mes free tier).
2. **Add cluster** → tipo **Basic** → región más cercana → **Launch cluster** (~1 min).
3. Dentro del cluster, menú **Topics** → **Add topic** → nombre `bookings-events` → 1 partición → **Create**.
4. Menú **API Keys** → **Add key** → Global access → Copiar **Key** y **Secret** (no se vuelven a mostrar).
5. Menú **Cluster settings** → copiar **Bootstrap server** (algo como `pkc-xxxx.us-east-1.aws.confluent.cloud:9092`).

Las 3 cadenas que necesitas pegar abajo: Bootstrap server, API Key, API Secret.

## 1. Credenciales y configuración

> ⚠️ En producción esto iría en Databricks Secrets (`dbutils.secrets.get(...)`). Para el proyecto académico se dejan inline.

In [ ]:
# === Pega aquí las credenciales de Confluent Cloud ===
CONFLUENT_BOOTSTRAP = "pkc-XXXXX.region.provider.confluent.cloud:9092"
CONFLUENT_API_KEY    = "TU_API_KEY_AQUI"
CONFLUENT_API_SECRET = "TU_API_SECRET_AQUI"
TOPIC_NAME           = "bookings-events"

print("Bootstrap:", CONFLUENT_BOOTSTRAP)
print("Topic    :", TOPIC_NAME)

## 2. Instalación de la librería del productor

Se usa `confluent-kafka` (el cliente oficial). Solo se instala si no existe en el cluster.

In [ ]:
%pip install confluent-kafka --quiet

In [ ]:
dbutils.library.restartPython()

## 3. Productor sintético de eventos de reservas

Genera 100 eventos de reserva basados en usuarios y propiedades reales (tomados
de la capa Silver) para que los datos sean coherentes con el resto del pipeline.
Cada evento se publica como JSON en el topic `bookings-events`.

In [ ]:
import json
import random
import time
from datetime import datetime, timedelta
from confluent_kafka import Producer

# Variables capturadas al inicio del notebook
bootstrap = CONFLUENT_BOOTSTRAP
api_key   = CONFLUENT_API_KEY
api_secret = CONFLUENT_API_SECRET
topic     = TOPIC_NAME

# Muestras reales desde Silver para que los eventos sean válidos
users_sample = [r.user_id for r in spark.table("silver.silver_users").select("user_id").limit(500).collect()]
props_sample = [r.property_id for r in spark.table("silver.silver_properties").select("property_id").limit(500).collect()]

producer = Producer({
    "bootstrap.servers": bootstrap,
    "security.protocol": "SASL_SSL",
    "sasl.mechanism":    "PLAIN",
    "sasl.username":     api_key,
    "sasl.password":     api_secret,
    "client.id":         "wanderbricks-producer"
})

def delivery_report(err, msg):
    if err is not None:
        print(f"❌ Error: {err}")

N_EVENTOS = 100
print(f"Publicando {N_EVENTOS} eventos en topic '{topic}'...")

for i in range(N_EVENTOS):
    check_in = datetime.now().date() + timedelta(days=random.randint(1, 90))
    nights   = random.randint(1, 14)
    evento = {
        "booking_id":   200000 + i,
        "user_id":      random.choice(users_sample),
        "property_id":  random.choice(props_sample),
        "check_in":     check_in.isoformat(),
        "check_out":    (check_in + timedelta(days=nights)).isoformat(),
        "guests_count": random.randint(1, 6),
        "total_amount": round(random.uniform(50, 1500), 2),
        "status":       random.choice(["pending", "confirmed", "cancelled"]),
        "created_at":   datetime.now().isoformat(),
        "updated_at":   datetime.now().isoformat()
    }
    producer.produce(
        topic,
        key   = str(evento["booking_id"]),
        value = json.dumps(evento),
        callback = delivery_report
    )
    if i % 20 == 0:
        producer.poll(0)
        print(f"  enviados: {i+1}/{N_EVENTOS}")
    time.sleep(0.05)

producer.flush()
print(f"✅ {N_EVENTOS} eventos publicados")

## 4. Consumidor con Spark Structured Streaming

Lee del topic en tiempo real, parsea el JSON al esquema esperado y escribe en una
tabla Delta `bronze.bronze_bookings_stream`. Se usa `checkpointLocation` para que
el consumo sea tolerante a fallos (exactly-once).

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DoubleType

schema_evento = StructType([
    StructField("booking_id",   LongType()),
    StructField("user_id",      LongType()),
    StructField("property_id",  LongType()),
    StructField("check_in",     StringType()),
    StructField("check_out",    StringType()),
    StructField("guests_count", IntegerType()),
    StructField("total_amount", DoubleType()),
    StructField("status",       StringType()),
    StructField("created_at",   StringType()),
    StructField("updated_at",   StringType())
])

jaas = (
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="{CONFLUENT_API_KEY}" password="{CONFLUENT_API_SECRET}";'
)

stream_df = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers",  CONFLUENT_BOOTSTRAP)
         .option("kafka.security.protocol",  "SASL_SSL")
         .option("kafka.sasl.mechanism",     "PLAIN")
         .option("kafka.sasl.jaas.config",   jaas)
         .option("subscribe",                TOPIC_NAME)
         .option("startingOffsets",          "earliest")
         .load()
)

eventos = (
    stream_df
      .selectExpr("CAST(value AS STRING) AS json_str")
      .select(from_json(col("json_str"), schema_evento).alias("data"))
      .select("data.*")
)

query = (
    eventos.writeStream
           .format("delta")
           .outputMode("append")
           .option("checkpointLocation", "/tmp/checkpoints/bronze_bookings_stream")
           .trigger(processingTime="5 seconds")
           .toTable("bronze.bronze_bookings_stream")
)

print("Streaming activo. Esperando 30 segundos para procesar lotes...")
query.awaitTermination(timeout=30)
query.stop()
print("✅ Streaming detenido")

## 5. Validación — los eventos llegaron a Bronze

In [ ]:
%sql
SELECT COUNT(*) AS eventos_recibidos
FROM bronze.bronze_bookings_stream;

In [ ]:
%sql
SELECT *
FROM bronze.bronze_bookings_stream
ORDER BY created_at DESC
LIMIT 10;

## 6. Integración con Silver (opcional)

Los eventos del stream comparten esquema con `silver_bookings`, por lo que se
pueden unir y refrescar la capa Silver de forma incremental con un `MERGE INTO`
cuando se ejecute el notebook `04_silver_layer`. Esto valida que el pipeline
Bronze (streaming) → Silver → Gold cierra el ciclo completo.

In [ ]:
%sql
-- Vista unificada que combina la carga batch original + los eventos de streaming
CREATE OR REPLACE VIEW bronze.bronze_bookings_all AS
SELECT booking_id, user_id, property_id,
       CAST(check_in AS DATE)  AS check_in,
       CAST(check_out AS DATE) AS check_out,
       guests_count, total_amount, status,
       CAST(created_at AS TIMESTAMP) AS created_at,
       CAST(updated_at AS TIMESTAMP) AS updated_at,
       'batch'    AS origen
FROM bronze.bronze_bookings
UNION ALL
SELECT booking_id, user_id, property_id,
       CAST(check_in AS DATE)  AS check_in,
       CAST(check_out AS DATE) AS check_out,
       guests_count, total_amount, status,
       CAST(created_at AS TIMESTAMP) AS created_at,
       CAST(updated_at AS TIMESTAMP) AS updated_at,
       'streaming' AS origen
FROM bronze.bronze_bookings_stream;

SELECT origen, COUNT(*) AS registros
FROM bronze.bronze_bookings_all
GROUP BY origen;

## Conclusión

El notebook demuestra el flujo completo de streaming exigido por la guía:

1. **Productor** sintético publica eventos JSON en un topic Kafka real (Confluent Cloud).
2. **Spark Structured Streaming** consume el topic vía conector nativo de Databricks.
3. Los eventos se persisten en una tabla Delta `bronze.bronze_bookings_stream` con
   checkpoint para garantizar exactly-once.
4. Una vista `bronze_bookings_all` une la carga batch original con los eventos de
   streaming, lo que permite que las capas Silver y Gold sigan trabajando sin
   cambios sobre la fuente combinada.

Decisiones de diseño defendibles ante el docente:

- **¿Por qué Confluent Cloud y no Kafka local?** Free tier suficiente (5 GB/mes),
  sin infraestructura local, mismas APIs que Kafka productivo.
- **¿Por qué `from_json` en el consumer?** El payload viaja como JSON (formato
  recomendado por la guía); el esquema explícito evita inferencia y errores
  silenciosos.
- **¿Por qué `checkpointLocation`?** Garantiza exactly-once y permite reanudar
  el stream tras un fallo sin duplicar eventos.
- **¿Por qué `trigger(processingTime="5 seconds")`?** Micro-batches cortos para
  baja latencia, sin ser tan agresivos como continuous processing (más complejo
  de operar).